# Private CayleyPy Results Ingest npm gate
CPU-only dependency, test, and TypeScript verification. No deployment.


In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

WORKING = Path("/kaggle/working")
ROOT = WORKING / "cayleypy-results-ingest-gate"
PACKAGE = ROOT / "services" / "cayleypy-results-ingest"
NPM_CACHE = Path("/tmp/cayleypy-results-ingest-npm-cache")
PAYLOAD_B64 = '...'
EXPECTED_SHA256 = {'configs/cayleypy_results_schema_v1.json': '5dc4888c70836229e0fb31508cf967082a186e472091d58826450368c2a9f126', 'services/cayleypy-results-ingest/package.json': 'e55ba61809d63740bd7d4dd4c009e57f06f2c1b3346a4ad3f147812641a29863', 'services/cayleypy-results-ingest/package-lock.json': '6c84152419596a537b4f0df8dee1ab2e843103faf606506bd46a6483fcf10b81', 'services/cayleypy-results-ingest/tsconfig.json': '0000e32074d2a3730a337cc3291e247cddbef2ebd8f98772b83cfe48a538bfd0', 'services/cayleypy-results-ingest/wrangler.jsonc': 'fbca840524e5c93b7805418e5046f83bbc89fde2ecca272c99be2011c93625d7', 'services/cayleypy-results-ingest/vitest.config.ts': 'a63ad781e2a6832f52ab0d0ba0e6a2950595492fd12883fa4ecb71f212d790e6', 'services/cayleypy-results-ingest/vitest.schema.config.ts': 'e06f4a73c0c2a9a662c2a9e2485e0fda83fcdae435efa0b7207960fa7fdc2d9a', 'services/cayleypy-results-ingest/migrations/0001_initial.sql': '8c3fe6fdc4381e123a901593962194f90aae7bfc484e6a6f5685195d05cb0ba6', 'services/cayleypy-results-ingest/migrations/0002_ingest_rate_limits.sql': '803e8b3af0dc4e1af2ddd32301a14fdba9d543cf9d5512a934f2d581f85e28f4', 'services/cayleypy-results-ingest/src/schema.ts': '5b2d0cdc1736bb5af720624d6f069567975d06991821e6a22776eacc3199126f', 'services/cayleypy-results-ingest/src/ids.ts': 'beec2434b4d4c96eaf7aeef3258f938f2e1ca6c123389096987365ff8e6727a6', 'services/cayleypy-results-ingest/src/db.ts': 'd46481ee7eb8a7d2cbc3673f305566b31b6605eb4c6cae1d7547cfbec72a6609', 'services/cayleypy-results-ingest/src/storage.ts': '929a71468e5f5773e1c6aad8416360a93e46213b5f715819dd8bdd9fc9338c8b', 'services/cayleypy-results-ingest/src/worker.ts': 'f45a1fb11fb80c93a872e20645faa3c7e822075b640347b64f3be8a93259dc11', 'services/cayleypy-results-ingest/test/schema.test.ts': '28ff4675db6dc1be505e9d9591f0491e84dab8d4cf6b7fad24895fab79f4900c', 'services/cayleypy-results-ingest/test/apply-migrations.ts': 'f7cc52a868ed3bf5ae2cf93fead32335a1541019d7828e16eb1375b0ebe5918d', 'services/cayleypy-results-ingest/test/receipt.test.ts': '686b3beb31adb722f1647bbd8a9ab95807edb06419222178cfe436eb1b2e6094', 'services/cayleypy-results-ingest/test/worker.test.ts': '4a3f514a3e33f47599aa529c72e175d1211d3b74962518c0105dee1c89363ddc'}

if ROOT.exists():
    shutil.rmtree(ROOT)
if NPM_CACHE.exists():
    shutil.rmtree(NPM_CACHE)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD_B64))) as archive:
    archive.extractall(ROOT)

observed = {}
for relative in EXPECTED_SHA256:
    observed[relative] = hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
if observed != EXPECTED_SHA256:
    raise RuntimeError("embedded payload checksum mismatch")
(WORKING / "payload-sha256.json").write_text(
    json.dumps(observed, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

combined_log = WORKING / "npm-gate.log"
combined_log.write_text("", encoding="utf-8")
results = {"payload_sha256": observed, "commands": []}

def run(label: str, argv: list[str]) -> int:
    completed = subprocess.run(
        argv,
        cwd=PACKAGE,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, "NPM_CONFIG_CACHE": str(NPM_CACHE)},
    )
    output = completed.stdout
    section = f"\n===== {label} (exit={completed.returncode}) =====\n{output}"
    print(section, flush=True)
    with combined_log.open("a", encoding="utf-8") as handle:
        handle.write(section)
    (WORKING / f"{label}.log").write_text(output, encoding="utf-8")
    results["commands"].append(
        {"label": label, "argv": argv, "exit_code": completed.returncode}
    )
    return completed.returncode

run("node-version", ["node", "--version"])
run("npm-version", ["npm", "--version"])
run("npm-install-package-lock-only", ["npm", "install", "--package-lock-only", "--no-audit", "--no-fund"])
run("npm-ci", ["npm", "ci", "--no-audit", "--no-fund"])
run("npm-test", ["npm", "test"])
run("npm-test-worker", ["npm", "exec", "--", "vitest", "run", "--config", "vitest.config.ts"])
run("npm-typecheck", ["npm", "run", "typecheck"])

lockfile = PACKAGE / "package-lock.json"
if lockfile.is_file():
    shutil.copy2(lockfile, WORKING / "package-lock.json")
results["lockfile_present"] = lockfile.is_file()
results["all_commands_passed"] = all(item["exit_code"] == 0 for item in results["commands"])
(WORKING / "npm-gate-results.json").write_text(
    json.dumps(results, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
shutil.rmtree(ROOT)
shutil.rmtree(NPM_CACHE)
print(json.dumps(results, indent=2, sort_keys=True), flush=True)
